# 🚀 اجرای مدل Hermes 3 (8B) + رابط کاربری Open WebUI در Google Colab

این نوت‌بوک به شما اجازه می‌دهد تا بدون نیاز به پرداخت هزینه یا سرور اختصاصی، مدل قدرتمند **Hermes 3 (8B)** را به همراه پنل کاربری شبیه به ChatGPT (**Open WebUI**) بر روی پردازنده گرافیکی رایگان گوگل کولب (T4 GPU) اجرا کرده و از طریق لینک اختصاصی Cloudflare Tunnel به آن متصل شوید.

### گام ۱: بررسی وضعیت کارت گرافیک (GPU)
مطمئن شوید در منوی بالای کولب مسیر Runtime > Change runtime type را باز کرده و گزینه **T4 GPU** را فعال کرده‌اید.

In [ ]:
!nvidia-smi

### گام ۲: نصب پیش‌نیازها و نرم‌افزار Ollama

In [ ]:
# نصب پیش‌نیازها و آخرین نسخه Ollama
!apt-get update -qq && apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh

### گام ۳: اجرای سرویس Ollama در پس‌زمینه

In [ ]:
import subprocess
import time
import requests

print("در حال اجرای سرویس Ollama...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

# بررسی سلامت و بالا آمدن سرویس
for _ in range(30):
    try:
        response = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        if response.status_code == 200:
            print("✅ Ollama با موفقیت راه‌اندازی شد و آماده به کار است!")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("خطا: سرویس Ollama راه‌اندازی نشد.")

### گام ۴: دانلود مدل Hermes 3 (8B)

In [ ]:
!ollama pull hermes3:8b

### گام ۵: تست چت و پاسخ‌دهی به زبان فارسی

In [ ]:
import requests

print("در حال ارسال پیام تست به مدل هرمس...")
response = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json={
        "model": "hermes3:8b",
        "messages": [
            {"role": "user", "content": "سلام! خودت را معرفی کن و فارسی جواب بده."}
        ],
        "stream": False
    },
    timeout=300
)

response.raise_for_status()
print("
🤖 پاسخ مدل:")
print(response.json()["message"]["content"])

### گام ۶: نصب پنل گرافیکی Open WebUI

In [ ]:
%pip install -q open-webui

### گام ۷: اجرای Open WebUI

In [ ]:
import os
import subprocess
import time
import requests

os.environ["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"
os.environ["WEBUI_AUTH"] = "False"  # ورود مستقیم بدون نیاز به ثبت‌نام در هر اجرا

print("در حال راه‌اندازی Open WebUI (حدود ۳۰ تا ۵۰ ثانیه زمان می‌برد)...")
webui_process = subprocess.Popen(
    ["open-webui", "serve", "--host", "0.0.0.0", "--port", "8080"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

for _ in range(60):
    try:
        response = requests.get("http://127.0.0.1:8080/", timeout=3)
        if response.status_code < 500:
            print("✅ Open WebUI با موفقیت روی پورت 8080 اجرا شد!")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("خطا در راه‌اندازی Open WebUI. لاگ‌ها را بررسی کنید.")

### گام ۸: ساخت تونل Cloudflare و دریافت لینک دسترسی عمومی

In [ ]:
# دانلود و نصب ابزار cloudflared در صورت عدم وجود
!which cloudflared > /dev/null || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb)

import subprocess
import re
import time

print("در حال اتصال به شبکه کلودفلر و تولید لینک اختصاصی...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in tunnel_process.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print("
" + "="*60)
        print(f"🎉 پنل هوش مصنوعی شما آماده است! روی لینک زیر بزنید:
👉 {tunnel_url}")
        print("="*60 + "
")
        break

# نگه داشتن فعال بودن تونل
try:
    tunnel_process.wait()
except KeyboardInterrupt:
    tunnel_process.terminate()
    print("تونل متوقف شد.")